# Implement feature


## Import


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# log import setup

from loguru import logger as lg
from rich import get_console
from rich import print as rprint
from rich.console import Console

# some magic to make rich work in jupyter
# https://github.com/Textualize/rich/issues/3483
# enable it for every cell output with %load_ext rich
console: Console = get_console()
console.is_jupyter = False

In [ ]:
# ADD YOUR IMPORTS HERE, or in later cells

from lang_tools.params.lang_tools_params import get_lang_tools_params


## Params and config


In [ ]:
# load env vars from cred file

from lang_tools.params.load_env import load_env

load_env()

In [ ]:
lang_tools_params = get_lang_tools_params()
rprint(lang_tools_params)

## Develop and prototype


# Phase 3 - storage & indexing experiments

Measures the two axes from [`03_storage_indexing.md`](../03_storage_indexing.md):

- **Axis A - distribution / storage format**: on-disk size per table for JSONL, JSONL+gzip, CSV (flat tables), Parquet (snappy / zstd), and SQLite. Drives the _normal-git-vs-LFS_ pivot.
- **Axis B - runtime access**: load latency + resident memory and point-lookup / filter latency for in-memory pydantic dicts (current store), SQLite (indexed), and DuckDB-over-Parquet, against phase 4's query surface (id point lookups, sense adjacency, symmetric false-friend fan-out, language+topic filter).

Data is **synthetic**, sized to the OMW 5-language estimate (~1e5 concepts, ~1e6 senses). Files are written to a temp dir outside the repo. Numbers feed the decision memo folded back into the brainstorm/tracking.


## Imports, scale config & helpers


In [ ]:
import csv
import gc
import gzip
import json
import random
import sqlite3
import string
import tempfile
import time
from pathlib import Path

import duckdb
import pyarrow as pa
import pyarrow.parquet as pq

from lang_tools.language.normalization import normalize
from lang_tools.lexicon.concept_id import concept_id
from lang_tools.lexicon.lemma import Lemma
from lang_tools.lexicon.lemma_id import lemma_id

# --- scale: OMW 5-language estimate (~1e5 concepts, ~1e6 senses) ---
LANGS = ["en", "es", "pt", "de", "fr"]
N_CONCEPTS = 100_000
N_LEMMAS = 400_000
N_SENSES = 1_000_000
N_FALSE_FRIENDS = 50_000
N_CONCEPT_RELATIONS = 200_000
SEED = 20260616

POS = ["noun", "verb", "adj", "adv", "pron", "prep"]
TOPICS = [
    "food",
    "travel",
    "work",
    "family",
    "nature",
    "science",
    "sport",
    "music",
    "health",
    "law",
]

BENCH_DIR = Path(tempfile.gettempdir()) / "p3_storage_bench"
BENCH_DIR.mkdir(exist_ok=True)


def rss_mb() -> float:
    """Current process resident set size in MB (from /proc/self/status)."""
    for line in Path("/proc/self/status").read_text().splitlines():
        if line.startswith("VmRSS:"):
            return int(line.split()[1]) / 1024
    return float("nan")


def sizeof_fmt(n: float) -> str:
    """Human-readable byte size."""
    x = float(n)
    for unit in ["B", "KB", "MB", "GB"]:
        if x < 1024:
            return f"{x:,.1f} {unit}"
        x /= 1024
    return f"{x:,.1f} TB"


console.print(
    f"bench dir: {BENCH_DIR}\nRSS now: {rss_mb():.0f} MB | "
    f"scale: concepts={N_CONCEPTS:,} lemmas={N_LEMMAS:,} senses={N_SENSES:,} "
    f"false_friends={N_FALSE_FRIENDS:,} concept_relations={N_CONCEPT_RELATIONS:,}",
)

## Synthetic dataset generator

Pure, seeded generators yield the **persisted dict shape** of each model (the lean shape: source fields + the `id` primary key, dropping the cosmetic computed fields `has_accent`/`accented_chars`/`length` that the store recomputes on load). Only the id pools are kept resident; every table is produced as a stream so serialization never holds a full table in RAM. Per-sense frequency/CEFR fields are left empty (phase 6 unpopulated), which is the realistic initial shape.


In [ ]:
# --- id pools (kept resident; small) ---
_t0 = time.perf_counter()
CONCEPT_IDS = [concept_id(f"concept-{i}", f"omw-{i:07d}") for i in range(N_CONCEPTS)]


def _suffix(i: int) -> str:
    """Base-26 encoding of i, so every lemma text is corpus-unique (no id collisions)."""
    s, x = "", i
    while True:
        x, d = divmod(x, 26)
        s += string.ascii_lowercase[d]
        if x == 0:
            return s


def _lemma_text(i: int) -> tuple[str, str]:
    """Deterministic, unique (text, language) for lemma index i."""
    r = random.Random(i)
    base = "".join(r.choices(string.ascii_lowercase, k=r.randint(3, 8)))
    return base + _suffix(i), LANGS[i % len(LANGS)]


LEMMA_IDS = [lemma_id(*_lemma_text(i)) for i in range(N_LEMMAS)]
assert len(set(LEMMA_IDS)) == N_LEMMAS, "lemma id collision"
lg.info(f"id pools built in {time.perf_counter() - _t0:.1f}s | RSS={rss_mb():.0f} MB")

_SENT = [
    "the",
    "cat",
    "runs",
    "fast",
    "over",
    "green",
    "hills",
    "every",
    "quiet",
    "morning",
    "with",
    "joy",
]


def gen_lemmas():
    for i in range(N_LEMMAS):
        r = random.Random(i)
        text, lang = _lemma_text(i)
        topics = [TOPICS[i % len(TOPICS)]]
        if i % 7 == 0:
            topics.append(TOPICS[(i * 3) % len(TOPICS)])
        examples = []
        if i % 5 == 0:  # ~20% of lemmas carry one curated example
            s = " ".join(r.choices(_SENT, k=10))
            examples = [{"sentence": s, "translation": s[::-1]}]
        yield {
            "id": LEMMA_IDS[i],
            "text": text,
            "language": lang,
            "normalized": normalize(text),
            "part_of_speech": POS[i % len(POS)],
            "topics": topics,
            "examples": examples,
            "sources": ["omw"],
        }


def gen_concepts():
    for i in range(N_CONCEPTS):
        defs = {"en": f"definition number {i} describing the concept in a few words"}
        if i % 3 == 0:
            defs["es"] = f"definicion numero {i} que describe el concepto brevemente"
        yield {"id": CONCEPT_IDS[i], "definitions": defs}


def gen_senses():
    r = random.Random(SEED + 1)
    for i in range(N_SENSES):
        yield {
            "id": f"{i:016x}",
            "lemma_id": LEMMA_IDS[r.randrange(N_LEMMAS)],
            "concept_id": CONCEPT_IDS[r.randrange(N_CONCEPTS)],
            "token_frequency": None,
            "sense_frequency": None,
            "frequency_is_estimated": False,
            "cefr_level": None,
            "cefr_is_estimated": False,
        }


def gen_false_friends():
    r = random.Random(SEED + 2)
    for _ in range(N_FALSE_FRIENDS):
        a, b = LEMMA_IDS[r.randrange(N_LEMMAS)], LEMMA_IDS[r.randrange(N_LEMMAS)]
        if a == b:
            continue
        if a > b:
            a, b = b, a
        yield {
            "lemma_id_a": a,
            "lemma_id_b": b,
            "similarity_score": round(r.random(), 3),
            "explanation_notes": {},
        }


def gen_concept_relations():
    r = random.Random(SEED + 3)
    rels = ["hypernym", "hyponym", "meronym", "related"]
    for _ in range(N_CONCEPT_RELATIONS):
        a, b = (
            CONCEPT_IDS[r.randrange(N_CONCEPTS)],
            CONCEPT_IDS[r.randrange(N_CONCEPTS)],
        )
        if a == b:
            continue
        yield {"concept_id_a": a, "concept_id_b": b, "relation_type": r.choice(rels)}


lg.info("generators ready")

## Axis A - serialization & on-disk size

Writers stream records (batched for Parquet/SQLite) so peak memory stays bounded. CSV is only applied to the flat tables (`senses`, `concept_relations`); the nested tables (lemmas/concepts/false_friends with lists/maps) would require JSON-in-cell, which defeats CSV diffability. Parquet stores nested columns natively (`list`/`struct`/`map`).


In [ ]:
BATCH = 50_000


def write_jsonl(path, records, *, gzipped=False):
    op = gzip.open if gzipped else open
    with op(path, "wt", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False))
            f.write("\n")
    return path.stat().st_size


def write_csv(path, records, fieldnames, *, gzipped=False):
    op = gzip.open if gzipped else open
    with op(path, "wt", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for r in records:
            w.writerow(r)
    return path.stat().st_size


def write_parquet(path, records, schema, compression):
    writer = pq.ParquetWriter(path, schema, compression=compression)
    batch = []
    for r in records:
        batch.append(r)
        if len(batch) >= BATCH:
            writer.write_table(pa.Table.from_pylist(batch, schema=schema))
            batch.clear()
    if batch:
        writer.write_table(pa.Table.from_pylist(batch, schema=schema))
    writer.close()
    return path.stat().st_size


def write_sqlite(path, table, columns, json_cols, records, index_ddls):
    if path.exists():
        path.unlink()
    con = sqlite3.connect(path)
    con.execute(f"CREATE TABLE {table} ({', '.join(columns)})")
    sql = f"INSERT INTO {table} VALUES ({','.join('?' * len(columns))})"
    cur = con.cursor()
    buf = []
    for r in records:
        buf.append(
            tuple(
                json.dumps(r[c], ensure_ascii=False) if c in json_cols else r[c]
                for c in columns
            )
        )
        if len(buf) >= BATCH:
            cur.executemany(sql, buf)
            buf.clear()
    if buf:
        cur.executemany(sql, buf)
    for ddl in index_ddls:
        con.execute(ddl)
    con.commit()
    con.execute("VACUUM")
    con.commit()
    con.close()
    return path.stat().st_size


_ex = pa.struct([("sentence", pa.string()), ("translation", pa.string())])
SCHEMAS = {
    "lemmas": pa.schema(
        [
            ("id", pa.string()),
            ("text", pa.string()),
            ("language", pa.string()),
            ("normalized", pa.string()),
            ("part_of_speech", pa.string()),
            ("topics", pa.list_(pa.string())),
            ("examples", pa.list_(_ex)),
            ("sources", pa.list_(pa.string())),
        ]
    ),
    "concepts": pa.schema(
        [("id", pa.string()), ("definitions", pa.map_(pa.string(), pa.string()))]
    ),
    "senses": pa.schema(
        [
            ("id", pa.string()),
            ("lemma_id", pa.string()),
            ("concept_id", pa.string()),
            ("token_frequency", pa.float64()),
            ("sense_frequency", pa.float64()),
            ("frequency_is_estimated", pa.bool_()),
            ("cefr_level", pa.string()),
            ("cefr_is_estimated", pa.bool_()),
        ]
    ),
    "false_friends": pa.schema(
        [
            ("lemma_id_a", pa.string()),
            ("lemma_id_b", pa.string()),
            ("similarity_score", pa.float64()),
            ("explanation_notes", pa.map_(pa.string(), pa.string())),
        ]
    ),
    "concept_relations": pa.schema(
        [
            ("concept_id_a", pa.string()),
            ("concept_id_b", pa.string()),
            ("relation_type", pa.string()),
        ]
    ),
}
lg.info("writers + schemas ready")

In [ ]:
# Per-table spec: generator, whether CSV is a clean fit, SQLite columns, JSON-encoded cols, indexes.
TABLE_SPECS = [
    {
        "name": "lemmas",
        "gen": gen_lemmas,
        "csv_ok": False,
        "csv_fields": None,
        "cols": [
            "id",
            "text",
            "language",
            "normalized",
            "part_of_speech",
            "topics",
            "examples",
            "sources",
        ],
        "json_cols": {"topics", "examples", "sources"},
        "indexes": [
            "CREATE UNIQUE INDEX ix_lemmas_id ON lemmas(id)",
            "CREATE INDEX ix_lemmas_lang ON lemmas(language)",
        ],
    },
    {
        "name": "concepts",
        "gen": gen_concepts,
        "csv_ok": False,
        "csv_fields": None,
        "cols": ["id", "definitions"],
        "json_cols": {"definitions"},
        "indexes": ["CREATE UNIQUE INDEX ix_concepts_id ON concepts(id)"],
    },
    {
        "name": "senses",
        "gen": gen_senses,
        "csv_ok": True,
        "csv_fields": [
            "id",
            "lemma_id",
            "concept_id",
            "token_frequency",
            "sense_frequency",
            "frequency_is_estimated",
            "cefr_level",
            "cefr_is_estimated",
        ],
        "cols": [
            "id",
            "lemma_id",
            "concept_id",
            "token_frequency",
            "sense_frequency",
            "frequency_is_estimated",
            "cefr_level",
            "cefr_is_estimated",
        ],
        "json_cols": set(),
        "indexes": [
            "CREATE INDEX ix_senses_lemma ON senses(lemma_id)",
            "CREATE INDEX ix_senses_concept ON senses(concept_id)",
        ],
    },
    {
        "name": "false_friends",
        "gen": gen_false_friends,
        "csv_ok": False,
        "csv_fields": None,
        "cols": ["lemma_id_a", "lemma_id_b", "similarity_score", "explanation_notes"],
        "json_cols": {"explanation_notes"},
        "indexes": [
            "CREATE INDEX ix_ff_a ON false_friends(lemma_id_a)",
            "CREATE INDEX ix_ff_b ON false_friends(lemma_id_b)",
        ],
    },
    {
        "name": "concept_relations",
        "gen": gen_concept_relations,
        "csv_ok": True,
        "csv_fields": ["concept_id_a", "concept_id_b", "relation_type"],
        "cols": ["concept_id_a", "concept_id_b", "relation_type"],
        "json_cols": set(),
        "indexes": ["CREATE INDEX ix_cr_a ON concept_relations(concept_id_a)"],
    },
]

sizes = {}
_t0 = time.perf_counter()
for spec in TABLE_SPECS:
    name, gen = spec["name"], spec["gen"]
    s = {}
    s["jsonl"] = write_jsonl(BENCH_DIR / f"{name}.jsonl", gen())
    s["jsonl.gz"] = write_jsonl(BENCH_DIR / f"{name}.jsonl.gz", gen(), gzipped=True)
    if spec["csv_ok"]:
        s["csv"] = write_csv(BENCH_DIR / f"{name}.csv", gen(), spec["csv_fields"])
        s["csv.gz"] = write_csv(
            BENCH_DIR / f"{name}.csv.gz", gen(), spec["csv_fields"], gzipped=True
        )
    s["parquet.snappy"] = write_parquet(
        BENCH_DIR / f"{name}.snappy.parquet", gen(), SCHEMAS[name], "snappy"
    )
    s["parquet.zstd"] = write_parquet(
        BENCH_DIR / f"{name}.zstd.parquet", gen(), SCHEMAS[name], "zstd"
    )
    s["sqlite"] = write_sqlite(
        BENCH_DIR / f"{name}.sqlite",
        name,
        spec["cols"],
        spec["json_cols"],
        gen(),
        spec["indexes"],
    )
    sizes[name] = s
    lg.info(
        f"{name}: "
        + " | ".join(f"{k}={sizeof_fmt(v)}" for k, v in s.items())
        + f" | RSS={rss_mb():.0f}MB"
    )
    gc.collect()
lg.info(f"serialization done in {time.perf_counter() - _t0:.1f}s")

In [ ]:
# Single combined SQLite DB (all tables + indexes) - the 'one runtime store file' option.
combined = BENCH_DIR / "all_tables.sqlite"
if combined.exists():
    combined.unlink()
_con = sqlite3.connect(combined)
for spec in TABLE_SPECS:
    name = spec["name"]
    _con.execute(f"CREATE TABLE {name} ({', '.join(spec['cols'])})")
    sql = f"INSERT INTO {name} VALUES ({','.join('?' * len(spec['cols']))})"
    buf = []
    for r in spec["gen"]():
        buf.append(
            tuple(
                json.dumps(r[c], ensure_ascii=False) if c in spec["json_cols"] else r[c]
                for c in spec["cols"]
            )
        )
        if len(buf) >= BATCH:
            _con.executemany(sql, buf)
            buf.clear()
    if buf:
        _con.executemany(sql, buf)
    for ddl in spec["indexes"]:
        _con.execute(ddl)
    _con.commit()
_con.execute("VACUUM")
_con.commit()
_con.close()
lg.info(
    f"combined sqlite (all tables): {sizeof_fmt(combined.stat().st_size)} | RSS={rss_mb():.0f}MB"
)

In [ ]:
from rich.table import Table

FORMATS = [
    "jsonl",
    "jsonl.gz",
    "csv",
    "csv.gz",
    "parquet.snappy",
    "parquet.zstd",
    "sqlite",
]
GH_WARN = 50 * 1024**2  # GitHub soft warning
GH_HARD = 100 * 1024**2  # GitHub hard push limit

tbl = Table(title="Axis A - on-disk size per table per format")
tbl.add_column("table")
tbl.add_column("rows", justify="right")
for f in FORMATS:
    tbl.add_column(f, justify="right")

rowcounts = {
    "lemmas": N_LEMMAS,
    "concepts": N_CONCEPTS,
    "senses": N_SENSES,
    "false_friends": N_FALSE_FRIENDS,
    "concept_relations": N_CONCEPT_RELATIONS,
}
totals = dict.fromkeys(FORMATS, 0)
for name, s in sizes.items():
    cells = [name, f"{rowcounts[name]:,}"]
    for f in FORMATS:
        v = s.get(f)
        if v is None:
            cells.append("-")
        else:
            totals[f] += v
            mark = " ⚠" if v >= GH_WARN else ""
            cells.append(sizeof_fmt(v) + mark)
    tbl.add_row(*cells)
tbl.add_row(
    *(["TOTAL", ""] + [sizeof_fmt(totals[f]) if totals[f] else "-" for f in FORMATS]),
    style="bold",
)
console.print(tbl)
console.print(
    f"⚠ = exceeds GitHub {sizeof_fmt(GH_WARN)} normal-git warning. Hard push limit {sizeof_fmt(GH_HARD)}."
)
console.print(
    f"single combined SQLite (all tables): {sizeof_fmt(combined.stat().st_size)}"
)

## Axis B - load latency & resident memory

Three access engines: in-memory pydantic dicts (the current store), SQLite (lazy, indexed), DuckDB-over-Parquet. Pydantic resident cost is measured directly for `lemmas`/`concepts` and extrapolated for the 1M `senses` from a measured subset (avoids OOM on this box).


In [ ]:
def load_pydantic_lemmas():
    """Current-store path: JSONL -> dict[id]=Lemma. Returns (dict, seconds, rss_delta_mb)."""
    gc.collect()
    before = rss_mb()
    t0 = time.perf_counter()
    by_id = {}
    with (BENCH_DIR / "lemmas.jsonl").open(encoding="utf-8") as f:
        for line in f:
            m = Lemma(**json.loads(line))
            by_id[m.id] = m
    dt = time.perf_counter() - t0
    return by_id, dt, rss_mb() - before


LEMMAS_BY_ID, lemma_load_s, lemma_rss = load_pydantic_lemmas()
lg.info(
    f"pydantic lemmas: {len(LEMMAS_BY_ID):,} loaded in {lemma_load_s:.1f}s, +{lemma_rss:.0f}MB RSS"
)

# Extrapolate pydantic Sense resident cost from a 100k subset.
from lang_tools.lexicon.sense import Sense

_SUB = 100_000
gc.collect()
_before = rss_mb()
_sub = []
for i, r in enumerate(gen_senses()):
    if i >= _SUB:
        break
    _sub.append(Sense(**r))
_sense_rss_sub = rss_mb() - _before
sense_rss_full = _sense_rss_sub / _SUB * N_SENSES
lg.info(
    f"pydantic senses subset {_SUB:,}: +{_sense_rss_sub:.0f}MB -> extrapolated {N_SENSES:,}: ~{sense_rss_full:,.0f}MB"
)
del _sub
gc.collect()

In [ ]:
# SQLite + DuckDB open cost (lazy engines).
gc.collect()
_b = rss_mb()
t0 = time.perf_counter()
sql_con = sqlite3.connect(f"file:{combined}?mode=ro", uri=True)
sql_open_s = time.perf_counter() - t0
sql_open_rss = rss_mb() - _b

gc.collect()
_b = rss_mb()
t0 = time.perf_counter()
duck = duckdb.connect(":memory:")
for name in rowcounts:
    duck.execute(
        f"CREATE VIEW {name} AS SELECT * FROM read_parquet('{BENCH_DIR / f'{name}.snappy.parquet'}')"
    )
duck_open_s = time.perf_counter() - t0
duck_open_rss = rss_mb() - _b

lg.info(
    f"sqlite open: {sql_open_s * 1000:.1f}ms, +{sql_open_rss:.1f}MB (lazy, indexed)"
)
lg.info(
    f"duckdb connect + {len(rowcounts)} parquet views: {duck_open_s * 1000:.1f}ms, +{duck_open_rss:.1f}MB (lazy)"
)

## Point-lookup & filter latency

The phase-4 query surface: `get_lemma_by_id` (PK point lookup), `get_false_friends_for_lemma` (symmetric adjacency fan-out), and `get_lemmas_filtered` by language+topic. Each timed across the in-memory dict, SQLite (indexed), and DuckDB-over-Parquet engines.


In [ ]:
N_Q = 2000
_r = random.Random(99)
query_ids = [LEMMA_IDS[_r.randrange(N_LEMMAS)] for _ in range(N_Q)]


def bench(fn, args, n_inner=1):
    t0 = time.perf_counter()
    for a in args:
        for _ in range(n_inner):
            fn(a)
    return (time.perf_counter() - t0) / (len(args) * n_inner) * 1e6  # microseconds/op


# --- get_lemma_by_id ---
dict_get = bench(lambda k: LEMMAS_BY_ID.get(k), query_ids)
_cur = sql_con.cursor()
sql_get = bench(
    lambda k: _cur.execute("SELECT * FROM lemmas WHERE id=?", (k,)).fetchone(),
    query_ids,
)
duck_get = bench(
    lambda k: duck.execute("SELECT * FROM lemmas WHERE id=?", [k]).fetchone(),
    query_ids[:200],
)

lg.info(
    f"get_lemma_by_id  dict={dict_get:.2f}us  sqlite={sql_get:.2f}us  duckdb={duck_get:.1f}us"
)

In [ ]:
# --- false-friends-by-lemma (symmetric adjacency) ---
FF_BY_LEMMA = {}
ff_pairs = []
for e in gen_false_friends():
    ff_pairs.append((e["lemma_id_a"], e["lemma_id_b"]))
    FF_BY_LEMMA.setdefault(e["lemma_id_a"], []).append(e)
    FF_BY_LEMMA.setdefault(e["lemma_id_b"], []).append(e)
ff_query = [p[0] for p in random.Random(7).sample(ff_pairs, 500)]

dict_ff = bench(lambda k: FF_BY_LEMMA.get(k, []), ff_query)
sql_ff = bench(
    lambda k: _cur.execute(
        "SELECT * FROM false_friends WHERE lemma_id_a=? OR lemma_id_b=?", (k, k)
    ).fetchall(),
    ff_query,
)
duck_ff = bench(
    lambda k: duck.execute(
        "SELECT * FROM false_friends WHERE lemma_id_a=? OR lemma_id_b=?", [k, k]
    ).fetchall(),
    ff_query[:200],
)
lg.info(
    f"false_friends_for_lemma  dict={dict_ff:.2f}us  sqlite={sql_ff:.2f}us  duckdb={duck_ff:.1f}us"
)

# --- filter lemmas by language + topic ---
FILTER_IDX = {}
for m in LEMMAS_BY_ID.values():
    for t in m.topics:
        FILTER_IDX.setdefault((m.language, t), []).append(m)
filter_keys = [
    ("en", "food"),
    ("es", "travel"),
    ("de", "work"),
    ("fr", "science"),
    ("pt", "music"),
]

dict_filt = bench(lambda k: FILTER_IDX.get(k, []), filter_keys, n_inner=50)
sql_filt = bench(
    lambda k: _cur.execute(
        "SELECT * FROM lemmas WHERE language=? AND topics LIKE ?", (k[0], f'%"{k[1]}"%')
    ).fetchall(),
    filter_keys,
    n_inner=5,
)
duck_filt = bench(
    lambda k: duck.execute(
        "SELECT * FROM lemmas WHERE language=? AND list_contains(topics, ?)",
        [k[0], k[1]],
    ).fetchall(),
    filter_keys,
    n_inner=5,
)
lg.info(
    f"filter lang+topic  dict={dict_filt:.2f}us  sqlite={sql_filt:.2f}us  duckdb={duck_filt:.1f}us"
)

In [ ]:
from rich.table import Table

# Resident-memory summary (pydantic full graph estimate).
concept_rss = 0.0
gc.collect()
_b = rss_mb()
from lang_tools.lexicon.concept import Concept

_cs = [Concept(**r) for r in gen_concepts()]
concept_rss = rss_mb() - _b
del _cs
gc.collect()

pydantic_total = lemma_rss + concept_rss + sense_rss_full

mem = Table(title="Axis B - resident memory & load (in-memory pydantic dicts)")
mem.add_column("table")
mem.add_column("rows", justify="right")
mem.add_column("resident", justify="right")
mem.add_column("note")
mem.add_row(
    "lemmas",
    f"{N_LEMMAS:,}",
    f"{lemma_rss:.0f} MB",
    f"measured, load {lemma_load_s:.1f}s",
)
mem.add_row("concepts", f"{N_CONCEPTS:,}", f"{concept_rss:.0f} MB", "measured")
mem.add_row(
    "senses",
    f"{N_SENSES:,}",
    f"~{sense_rss_full:,.0f} MB",
    "extrapolated from 100k subset",
)
mem.add_row(
    "FULL GRAPH",
    "",
    f"~{pydantic_total:,.0f} MB",
    "lemmas+concepts+senses as pydantic",
    style="bold",
)
console.print(mem)

lat = Table(title="Axis B - query latency (us/op; lower better)")
lat.add_column("query")
lat.add_column("in-mem dict", justify="right")
lat.add_column("sqlite (indexed)", justify="right")
lat.add_column("duckdb / parquet", justify="right")
lat.add_row("get_lemma_by_id", f"{dict_get:.2f}", f"{sql_get:.2f}", f"{duck_get:,.0f}")
lat.add_row(
    "false_friends_for_lemma", f"{dict_ff:.2f}", f"{sql_ff:.2f}", f"{duck_ff:,.0f}"
)
lat.add_row(
    "filter lang+topic", f"{dict_filt:.2f}", f"{sql_filt:.2f}", f"{duck_filt:,.0f}"
)
console.print(lat)
console.print(
    f"sqlite open +{sql_open_rss:.1f}MB / {sql_open_s * 1000:.1f}ms | "
    f"duckdb open +{duck_open_rss:.1f}MB / {duck_open_s * 1000:.1f}ms (both lazy, no full resident set)",
)

## Decision memo (measured)

**Scale:** 400k lemmas, 100k concepts, 1M senses, 50k false-friends, 200k concept-relations (the OMW 5-language estimate). Synthetic, lean persisted shape.

### Axis A - size (headline per-table, MB)

| table             | rows |     jsonl | parquet.zstd | sqlite |
| ----------------- | ---: | --------: | -----------: | -----: |
| lemmas            | 400k |    78.6 ⚠ |     **11.9** |   54.0 |
| concepts          | 100k |      14.5 |      **1.2** |   16.8 |
| senses            |   1M |   226.9 ⚠ |     **22.6** |  135.2 |
| false_friends     |  50k |       5.8 |      **1.0** |    4.9 |
| concept_relations | 200k |      24.6 |      **4.2** |   22.3 |
| **TOTAL**         |      | **350.3** |     **40.9** |  233.1 |

- **Parquet+zstd is ~8.6x smaller than JSONL** (40.9 MB vs 350.3 MB total) and beats snappy (72.5 MB).
- **The normal-git-vs-LFS pivot:** as JSONL, `senses` (227 MB) **exceeds GitHub's 100 MB hard push limit** and `lemmas` (79 MB) trips the 50 MB warning, so the big tables **need LFS regardless** - and since LFS does not delta-compress, JSONL's diff advantage is lost exactly where it would matter. As Parquet+zstd, _every_ per-table file is < 25 MB; per-language partitioning of `senses`/`lemmas` drops each to ~5 MB.

### Axis B - runtime access

Resident memory (in-memory pydantic dicts, the current store): **lemmas 745 MB (400k) + concepts 77 MB + senses ~1,075 MB (1M, extrapolated) = ~1.9 GB**. SQLite/DuckDB open lazily at ~0 MB resident.

Query latency (µs/op):

| query                   | in-mem dict | sqlite (indexed) | duckdb/parquet |
| ----------------------- | ----------: | ---------------: | -------------: |
| get_lemma_by_id         |         3.0 |               30 |         16,500 |
| false_friends_for_lemma |         0.6 |               34 |          8,900 |
| filter lang+topic       |         0.1 |          152,800 |        150,100 |

- **DuckDB is ~500x slower on point lookups** (16 ms) - confirmed _wrong_ for the hot per-lemma path; keep it as a build/QA tool over the shipped Parquet.
- **SQLite indexed point lookups (~30 µs) are imperceptible at ~0 resident memory** - the right runtime store once memory is the constraint.
- The **lang+topic filter is 150 ms** on both on-disk engines (LIKE / parquet scan); it needs an explicit **look-aside index** (in-memory adjacency dict, or a normalized topic table + index in SQLite), not a raw scan. Same lesson for sense adjacency.

### Decision (confirms / sharpens the provisional lean)

1. **Ship Parquet (zstd), partitioned per table and per language** for `senses`/`lemmas`, under git-LFS. Smallest objects, open/stable format, directly queryable by DuckDB for build/QA. **(confirmed)**
2. **Ship _all_ tables as Parquet under LFS, including the small curated ones, for uniformity** - _overturns_ the provisional "small tables as JSONL in normal git". One distribution path, no special cases (re-pointing the source later touches one mechanism); a 50k-row textual diff is not meaningfully reviewable, and a model change rewrites every JSONL line, so line-diffability is a false comfort. Inspect/edit is the explicit workflow below.
3. **Runtime: in-memory pydantic dicts only for the tiny bootstrap/sample data.** For the full OMW corpus, ~1.9 GB resident is too much (infeasible on a 512 MB Render dyno), so **promote the hot tables to SQLite indexed point lookups** (~30 µs, ~0 resident). This is now a _measured trigger_, not a "maybe later".
4. **Do not ship `.duckdb` or a single canonical `.sqlite`** as the artifact (version-stability / opaque-LFS-churn). DuckDB stays a build/QA reader; SQLite, if used at runtime, is a _built-from-Parquet_ index, not the shipped source of truth.
5. **Every filtered/adjacency access needs an explicit index** (look-aside dict or SQLite secondary index); a columnar/LIKE scan is 4-5 orders of magnitude slower than an indexed lookup.

### Drafted `.gitattributes` / partitioning

```gitattributes
# entire generated corpus -> LFS, one rule, no special cases
data/lexicon/**/*.parquet  filter=lfs diff=lfs merge=lfs -text
```

Partition the large tables as `data/lexicon/<table>/<lang>.parquet` so one language's re-ingest re-pushes only ~5 MB; small tables are a single Parquet each.

### Inspect / edit workflow (any table)

Nothing human-readable is committed, so inspection/editing are explicit operations over the canonical Parquet, routed through the phase-4 codec seam (`_load_table` / `_dump_table`):

- **Inspect (read-only):** DuckDB SQL directly over Parquet, no import step (`SELECT * FROM 'data/lexicon/senses/en.parquet' WHERE ...`); a thin `python -m lang_tools.lexicon.inspect <table> [--lang L] [--where SQL] [--limit N] [--format table|jsonl|csv]` CLI wraps it.
- **Edit (round-trip, validated):** `export_table(name, fmt="jsonl")` -> hand/LLM edit -> `import_table(name, path)` which **validates every row through the pydantic model** and rewrites the canonical Parquet. The JSONL is transient, never committed.
- **Schema changes are not edits:** adding/removing a column regenerates the table from the ingestion pipeline (phase 5), per the "no data migration" decision - never patched line-by-line.
